In [1]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = 'Magalong'
STUDENT_ID = 'TUPM-26-1134'

id_digits = ''.join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, 'Student ID must contain digits.'

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print('Student:', LAST_NAME, STUDENT_ID)
print('Seed Digit:', SEED_DIGIT)
print('ID Sum:', ID_SUM)
print('Name Length:', NAME_LENGTH)
print('Random Seed:', RANDOM_SEED)
print('Verification Value:', verification_value)

Student: Magalong TUPM-26-1134
Seed Digit: 4
ID Sum: 17
Name Length: 8
Random Seed: 261134
Verification Value: 140


In [2]:
# [CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / 'data'
OUTPUT_DIR = WORK_DIR / 'output'

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / 'secom.data'
label_path = DATA_DIR / 'secom_labels.data'
description_path = DATA_DIR / 'secom.names'

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f'Missing file: {path}'

print('Workspace:', WORK_DIR.resolve())
print('All dataset files verified.')

Workspace: C:\Users\Paul\Documents\TUPM-BSECE\1C\AECEL1\Lab\Lab7_NumPy_Pandas_Magalong\Lab7_Additionals
All dataset files verified.


In [3]:
# [CELL 3] LOAD DATA AND PREPARE OUTCOME GROUPS
import warnings

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw_measurements.ndim == 2
assert len(labels) == len(raw_measurements)
assert np.isin(labels, [-1, 1]).all()

pass_mask = labels == -1
fail_mask = labels == 1

print('Shape:', raw_measurements.shape)
print('Pass Records:', pass_mask.sum())
print('Fail Records:', fail_mask.sum())

Shape: (1567, 590)
Pass Records: 1463
Fail Records: 104


In [4]:
# [CELL 4] SELECT FIVE FEATURES WITH AT LEAST 10 OBSERVATIONS PER GROUP
missing = np.isnan(raw_measurements)
observed_pass = (~missing[pass_mask]).sum(axis=0)
observed_fail = (~missing[fail_mask]).sum(axis=0)
observed = (~missing).sum(axis=0)

observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)

eligible = np.flatnonzero(
    (observed_pass >= 10) & (observed_fail >= 10) & ~constant
)

rng = np.random.default_rng(RANDOM_SEED)
positions = rng.choice(len(eligible), size=5, replace=False)

selected_indices = eligible[positions]
selected_names = [f'Feature_{i:03d}' for i in selected_indices]

X = raw_measurements[:, selected_indices]

print('Eligible Features:', len(eligible))
print('Selected Features:', selected_names)
print('Observed in Pass:', observed_pass[selected_indices])
print('Observed in Fail:', observed_fail[selected_indices])

Eligible Features: 470
Selected Features: ['Feature_524', 'Feature_215', 'Feature_287', 'Feature_029', 'Feature_346']
Observed in Pass: [1463 1439 1460 1462  705]
Observed in Fail: [104 104 104 103  68]


In [5]:
# [CELL 5] PREPARATION STRATEGIES AND OBSERVED MEAN DIFFERENCES
# Strategy A: available-case analysis (missing values ignored per feature)
X_pass = X[pass_mask]
X_fail = X[fail_mask]

available_diff = np.nanmean(X_fail, axis=0) - np.nanmean(X_pass, axis=0)

# Strategy B: median imputation using medians from the full dataset
full_medians = np.nanmedian(X, axis=0)
X_imputed = np.where(np.isnan(X), full_medians, X)

X_pass_imp = X_imputed[pass_mask]
X_fail_imp = X_imputed[fail_mask]

imputed_diff = X_fail_imp.mean(axis=0) - X_pass_imp.mean(axis=0)

print('Full-Dataset Medians:', full_medians)
print('Available-Case Difference (Fail - Pass):', available_diff)
print('Median-Imputed Difference (Fail - Pass):', imputed_diff)

Full-Dataset Medians: [4.8771 0.0825 0.139  2.3778 3.0464]
Available-Case Difference (Fail - Pass): [-0.4844 -0.0022 -0.1423  0.0283 -0.1858]
Median-Imputed Difference (Fail - Pass): [-0.4844 -0.0023 -0.1418  0.0282  0.3497]


In [6]:
# [CELL 6] STRATIFIED BOOTSTRAP FUNCTION
n_boot = 200 + 10 * SEED_DIGIT
n_pass = X_pass.shape[0]
n_fail = X_fail.shape[0]

def stratified_bootstrap(seed):
    generator = np.random.default_rng(seed)

    # Independent resampling with replacement inside each outcome group
    idx_pass = generator.integers(0, n_pass, size=(n_boot, n_pass))
    idx_fail = generator.integers(0, n_fail, size=(n_boot, n_fail))

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        avail = (np.nanmean(X_fail[idx_fail], axis=1)
                 - np.nanmean(X_pass[idx_pass], axis=1))

    imputed = (X_fail_imp[idx_fail].mean(axis=1)
               - X_pass_imp[idx_pass].mean(axis=1))

    return avail, imputed

boot_available, boot_imputed = stratified_bootstrap(RANDOM_SEED)

print('Bootstrap Repetitions:', n_boot)
print('Available-Case Bootstrap Shape:', boot_available.shape)
print('Median-Imputed Bootstrap Shape:', boot_imputed.shape)

Bootstrap Repetitions: 240
Available-Case Bootstrap Shape: (240, 5)
Median-Imputed Bootstrap Shape: (240, 5)


In [7]:
# [CELL 7] BOOTSTRAP MEAN DIFFERENCES AND 95% CONFIDENCE INTERVALS
def summarize(boot):
    mean_diff = np.nanmean(boot, axis=0)
    lower, upper = np.nanpercentile(boot, [2.5, 97.5], axis=0)
    return mean_diff, lower, upper

avail_boot_mean, avail_low, avail_high = summarize(boot_available)
imp_boot_mean, imp_low, imp_high = summarize(boot_imputed)

for i, name in enumerate(selected_names):
    print(f'{name}')
    print(f'  Available-Case: {avail_boot_mean[i]:.4f} [{avail_low[i]:.4f}, {avail_high[i]:.4f}]')
    print(f'  Median-Imputed: {imp_boot_mean[i]:.4f} [{imp_low[i]:.4f}, {imp_high[i]:.4f}]')

Feature_524
  Available-Case: -0.4685 [-1.7274, 0.2930]
  Median-Imputed: -0.4685 [-1.7274, 0.2930]
Feature_215
  Available-Case: -0.0027 [-0.0079, 0.0034]
  Median-Imputed: -0.0027 [-0.0079, 0.0034]
Feature_287
  Available-Case: -0.1378 [-0.7321, 0.1012]
  Median-Imputed: -0.1373 [-0.7315, 0.1012]
Feature_029
  Available-Case: 0.0322 [-0.0455, 0.0968]
  Median-Imputed: 0.0321 [-0.0449, 0.0951]
Feature_346
  Available-Case: -0.2533 [-2.8459, 3.4898]
  Median-Imputed: 0.3150 [-1.1090, 2.6770]


In [8]:
# [CELL 8] COMPARE DIRECTION, INTERVALS AND RANKINGS
def rank_by_magnitude(values):
    # Rank 1 = largest absolute mean difference; ties keep feature order
    return np.argsort(np.argsort(-np.abs(values), kind='stable'), kind='stable') + 1

rank_available = rank_by_magnitude(available_diff)
rank_imputed = rank_by_magnitude(imputed_diff)

sign_changed = np.sign(available_diff) != np.sign(imputed_diff)
rank_changed = rank_available != rank_imputed

change_in_diff = np.abs(imputed_diff - available_diff)
largest_idx = int(np.argmax(change_in_diff))

avail_excludes_zero = (avail_low > 0) | (avail_high < 0)
imp_excludes_zero = (imp_low > 0) | (imp_high < 0)

print('Rank (Available-Case):', rank_available)
print('Rank (Median-Imputed):', rank_imputed)
print('Features with Changed Difference Sign:',
      [selected_names[i] for i in np.flatnonzero(sign_changed)])
print('Features with Changed Ranking:',
      [selected_names[i] for i in np.flatnonzero(rank_changed)])
print('Largest Change in Mean Difference:',
      selected_names[largest_idx], f'{change_in_diff[largest_idx]:.4f}')
print('CI Excludes Zero (Available-Case):', avail_excludes_zero)
print('CI Excludes Zero (Median-Imputed):', imp_excludes_zero)

Rank (Available-Case): [1 5 3 4 2]
Rank (Median-Imputed): [1 5 3 4 2]
Features with Changed Difference Sign: ['Feature_346']
Features with Changed Ranking: []
Largest Change in Mean Difference: Feature_346 0.5355
CI Excludes Zero (Available-Case): [False False False False False]
CI Excludes Zero (Median-Imputed): [False False False False False]


In [9]:
# [CELL 9] VERIFY REPRODUCIBILITY
repeat_available, repeat_imputed = stratified_bootstrap(RANDOM_SEED)

reproducible = (
    np.array_equal(boot_available, repeat_available, equal_nan=True)
    and np.array_equal(boot_imputed, repeat_imputed, equal_nan=True)
)

different_seed_available, _ = stratified_bootstrap(RANDOM_SEED + 1)
seed_matters = not np.array_equal(boot_available, different_seed_available, equal_nan=True)

print('Same Seed Reproduces Results:', reproducible)
print('Different Seed Changes Results:', seed_matters)

Same Seed Reproduces Results: True
Different Seed Changes Results: True


In [10]:
# [CELL 10] EXPORT COMPARISON AND SENSITIVITY REPORT
rows = []

for strategy, observed_diff, boot_mean, low, high, ranks, excludes in [
    ('Available-Case', available_diff, avail_boot_mean, avail_low, avail_high,
     rank_available, avail_excludes_zero),
    ('Median-Imputed', imputed_diff, imp_boot_mean, imp_low, imp_high,
     rank_imputed, imp_excludes_zero)
]:
    for i, name in enumerate(selected_names):
        rows.append({
            'Feature': name,
            'Strategy': strategy,
            'Observed_Mean_Difference': observed_diff[i],
            'Bootstrap_Mean_Difference': boot_mean[i],
            'CI_Lower_95': low[i],
            'CI_Upper_95': high[i],
            'CI_Excludes_Zero': bool(excludes[i]),
            'Rank': int(ranks[i])
        })

robustness = pd.DataFrame(rows)

comparison_path = OUTPUT_DIR / 'robustness_comparison.csv'
report_path = OUTPUT_DIR / 'sensitivity_report.txt'

robustness.to_csv(comparison_path, index=False)

changed_sign = [selected_names[i] for i in np.flatnonzero(sign_changed)]
changed_rank = [selected_names[i] for i in np.flatnonzero(rank_changed)]

report = (
    'SECOM STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS\n'
    f'Student: {LAST_NAME}\n'
    f'Student ID: {STUDENT_ID}\n'
    f'Random Seed: {RANDOM_SEED}\n'
    f'Verification Value: {verification_value}\n'
    f'Selected Features: {", ".join(selected_names)}\n'
    f'Bootstrap Repetitions: {n_boot}\n'
    f'Largest Change in Mean Difference: {selected_names[largest_idx]} '
    f'({change_in_diff[largest_idx]:.4f})\n'
    f'Features with Changed Difference Sign: {", ".join(changed_sign) if changed_sign else "None"}\n'
    f'Features with Changed Ranking: {", ".join(changed_rank) if changed_rank else "None"}\n'
    f'Reproducibility Verified: {reproducible}\n\n'
    'Interpretation Notes:\n'
    '- Available-case and median-imputed results can differ because imputation pulls missing values toward the full-dataset median.\n'
    '- A confidence interval containing zero means the observed Fail-Pass difference is not stable under resampling.\n'
    '- Rankings use raw mean differences, so features with larger measurement scales can rank higher.\n'
    '- Group differences do not establish failure causes.\n'
)

report_path.write_text(report, encoding='utf-8')

for path in [comparison_path, report_path]:
    print(path.name, 'Created:', path.is_file())

robustness_comparison.csv Created: True
sensitivity_report.txt Created: True


In [11]:
# [CELL 11] FINAL OUTPUT
print('ASSESSMENT DATA - STATISTICAL ROBUSTNESS AND SENSITIVITY')
print('--------------------------------------------------------')
print('Selected Features:', ', '.join(selected_names))
print('Bootstrap Repetitions:', n_boot)
print('Largest Change in Mean Difference:',
      selected_names[largest_idx], f'({change_in_diff[largest_idx]:.4f})')
print('Features with Changed Difference Sign:',
      ', '.join(changed_sign) if changed_sign else 'None')
print('Features with Changed Ranking:',
      ', '.join(changed_rank) if changed_rank else 'None')
print('Reproducibility Verified:', reproducible)
print('Required Outputs:', comparison_path.name, ',', report_path.name)

ASSESSMENT DATA - STATISTICAL ROBUSTNESS AND SENSITIVITY
--------------------------------------------------------
Selected Features: Feature_524, Feature_215, Feature_287, Feature_029, Feature_346
Bootstrap Repetitions: 240
Largest Change in Mean Difference: Feature_346 (0.5355)
Features with Changed Difference Sign: Feature_346
Features with Changed Ranking: None
Reproducibility Verified: True
Required Outputs: robustness_comparison.csv , sensitivity_report.txt
